In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
import statsmodels.formula.api as smf

In [2]:
def load_and_prepare_run(
    run_id,
    result_dir=""
):
    """
    Load loss, label, FOIF and TracIn results for one experimental run.

    Expected filenames:
        IF_sp1_de01_seed_<run_id>.csv
        TC_sp1_de01_seed_<run_id>.csv
        loss_sp1_de01_seed_<run_id>.csv
        label_sp1_de01_seed_<run_id>.csv
    """

    foif_df = pd.read_csv(
        f"IF_9_1_run{run_id}.csv"
    )
    tracin_df = pd.read_csv(
        f"TC_9_1_run{run_id}.csv"
    )
    loss_df = pd.read_csv(
        f"loss_9_1_run{run_id}.csv"
    )
    label_df = pd.read_csv(
        f"Class_label_9_1_run{run_id}.csv"
    )

    label_df = label_df[
        ["id", "label"]
    ].rename(columns={"id": "Train_ID"})

    foif_df = foif_df.rename(
        columns={"Score": "FOIF_Score"}
    )

    tracin_df = tracin_df.rename(
        columns={"Score": "TracIn_Score"}
    )

    density_df = (
        label_df
        .merge(loss_df, on="Train_ID", validate="one_to_one")
        .merge(foif_df, on="Train_ID", validate="one_to_one")
        .merge(tracin_df, on="Train_ID", validate="one_to_one")
    )


    density_df["Minority"] = density_df["label"].astype(int)

    density_df["Log_Loss"] = np.log1p(
        density_df["Training_Loss"]
    )

    density_df["Log_Abs_FOIF"] = np.log1p(
        np.abs(density_df["FOIF_Score"])
    )

    density_df["Run"] = run_id

    return density_df

In [3]:
def fit_regression_for_run(density_df, run_id):
    """
    Fit signed-score and magnitude regression models for one run.
    """

    signed_model = smf.ols(
        "FOIF_Score ~ Log_Loss * Minority",
        data=density_df
    ).fit()

    magnitude_model = smf.ols(
        "Log_Abs_FOIF ~ Log_Loss * Minority",
        data=density_df
    ).fit()

    signed_result = {
        "Run": run_id,
        "Model": "Signed FOIF",
        "Beta_0_Intercept": signed_model.params["Intercept"],
        "Beta_1_Log_Loss": signed_model.params["Log_Loss"],
        "Beta_2_Minority": signed_model.params["Minority"],
        "Beta_3_Interaction": signed_model.params["Log_Loss:Minority"],
        "Minority_Slope": (
            signed_model.params["Log_Loss"]
            + signed_model.params["Log_Loss:Minority"]
        ),
    }

    magnitude_result = {
        "Run": run_id,
        "Model": "Absolute FOIF magnitude",
        "Beta_0_Intercept": magnitude_model.params["Intercept"],
        "Beta_1_Log_Loss": magnitude_model.params["Log_Loss"],
        "Beta_2_Minority": magnitude_model.params["Minority"],
        "Beta_3_Interaction": magnitude_model.params[
            "Log_Loss:Minority"
        ],
        "Minority_Slope": (
            magnitude_model.params["Log_Loss"]
            + magnitude_model.params["Log_Loss:Minority"]
        ),
    }

    return signed_result, magnitude_result

In [4]:
def calculate_decile_summary(density_df, run_id):
    """
    Calculate loss-decile statistics for one experimental run.
    """

    density_df = density_df.copy()

    density_df["Loss_Decile"] = pd.qcut(
        density_df["Training_Loss"],
        q=10,
        labels=False,
        duplicates="drop"
    ) + 1

    decile_summary = (
        density_df
        .groupby(
            ["Loss_Decile", "Minority"],
            as_index=False
        )
        .agg(
            Count=("Train_ID", "size"),

            Mean_Loss=("Training_Loss", "mean"),
            Median_Loss=("Training_Loss", "median"),

            Mean_FOIF=("FOIF_Score", "mean"),
            FOIF_Positive_Fraction=(
                "FOIF_Score",
                lambda x: (x > 0).mean()
            ),

            Mean_TracIn=("TracIn_Score", "mean"),
            TracIn_Positive_Fraction=(
                "TracIn_Score",
                lambda x: (x > 0).mean()
            )
        )
    )

    decile_summary["Run"] = run_id

    return decile_summary

In [5]:
seeds = [1,2,3,4,5]

all_regression_results = []
all_run_data = []
all_median_loss_results = []
all_decile_results = []

for seed in seeds:
    print(f"Processing run {seed}...")

    density_df = load_and_prepare_run(
        run_id=seed,
        result_dir=""
    )

    # median_loss_run = (
    # density_df
    # .groupby("Minority")["Training_Loss"]
    # .median()
    # )

    # all_median_loss_results.append({
    #     "Run": seed,
    #     "Dense_Median_Loss": median_loss_run.get("Dense", np.nan),
    #     "Sparse_Median_Loss": median_loss_run.get("Sparse", np.nan)
    # })

    decile_summary_run = calculate_decile_summary(
        density_df=density_df,
        run_id=seed
    )

    all_decile_results.append(decile_summary_run)
    

    signed_result, magnitude_result = fit_regression_for_run(
        density_df=density_df,
        run_id=seed
    )

    all_regression_results.extend([
        signed_result,
        magnitude_result
    ])

    all_run_data.append(density_df)

regression_results_df = pd.DataFrame(
    all_regression_results
)

all_density_df = pd.concat(
    all_run_data,
    ignore_index=True
)

display(regression_results_df)

Processing run 1...
Processing run 2...
Processing run 3...
Processing run 4...
Processing run 5...


,Run,Model,Beta_0_Intercept,Beta_1_Log_Loss,Beta_2_Minority,Beta_3_Interaction,Minority_Slope
0,1,Signed FOIF,0.008752,-0.248779,0.055593,0.088566,-0.160213
1,1,Absolute FOIF magnitude,0.000270,0.194964,0.008519,-0.088708,0.106256
2,2,Signed FOIF,0.009004,-0.363303,0.021739,0.296798,-0.066505
3,2,Absolute FOIF magnitude,-0.001118,0.254941,0.011751,-0.172054,0.082887
4,3,Signed FOIF,0.008560,-0.333710,0.045647,0.164292,-0.169418
5,3,Absolute FOIF magnitude,-0.000161,0.249106,0.009750,-0.117568,0.131537
6,4,Signed FOIF,0.004944,-0.379601,0.015131,0.292016,-0.087585
7,4,Absolute FOIF magnitude,-0.000221,0.271537,0.007492,-0.165248,0.106289
8,5,Signed FOIF,0.005935,-0.257622,0.049636,0.148175,-0.109447
9,5,Absolute FOIF magnitude,-0.000579,0.213848,0.025022,-0.140103,0.073745


In [6]:
median_loss_results_df = pd.DataFrame(
    all_median_loss_results
)

display(median_loss_results_df)

""


In [7]:
coefficient_columns = [
    "Beta_0_Intercept",
    "Beta_1_Log_Loss",
    "Beta_2_Minority",
    "Beta_3_Interaction",
    "Minority_Slope",
]

regression_summary = (
    regression_results_df
    .groupby("Model")[coefficient_columns]
    .agg(["mean", "std"])
)

display(regression_summary)

Beta_0_Intercept           Beta_1_Log_Loss            \
                                    mean       std            mean       std   
Model                                                                          
Absolute FOIF magnitude        -0.000362  0.000519        0.236879  0.031481   
Signed FOIF                     0.007439  0.001866       -0.316603  0.060252   

                        Beta_2_Minority           Beta_3_Interaction  \
                                   mean       std               mean   
Model                                                                  
Absolute FOIF magnitude        0.012507  0.007174          -0.136736   
Signed FOIF                    0.037549  0.017957           0.197969   

                                  Minority_Slope            
                              std           mean       std  
Model                                                       
Absolute FOIF magnitude  0.034445       0.100143  0.022668  
Signed FOIF              0.092459      -0.118633  0.044927

In [8]:
decile_results_5_runs = pd.concat(
    all_decile_results,
    ignore_index=True
)

display(decile_results_5_runs)

,Loss_Decile,Minority,Count,Mean_Loss,Median_Loss,Mean_FOIF,FOIF_Positive_Fraction,Mean_TracIn,TracIn_Positive_Fraction,Run
0,1,0,798,0.000250,0.000249,0.000322,1.000000,-0.000371,0.154135,1
1,1,1,2,0.000313,0.000313,0.000758,1.000000,0.017388,1.000000,1
2,2,0,796,0.000808,0.000805,0.000936,1.000000,-0.000923,0.016332,1
3,2,1,4,0.000802,0.000802,0.001678,1.000000,0.022740,1.000000,1
4,3,0,795,0.001596,0.001569,0.001696,1.000000,-0.001505,0.001258,1
...,...,...,...,...,...,...,...,...,...,...
94,8,1,68,0.015544,0.015337,0.010148,1.000000,0.083730,1.000000,5
95,9,0,602,0.044617,0.037902,0.009690,0.943522,-0.028186,0.000000,5
96,9,1,198,0.053936,0.051547,0.032544,0.994949,0.090035,1.000000,5
97,10,0,351,0.538321,0.254041,-0.052987,0.404558,-0.056098,0.000000,5


In [9]:
statistics_to_summarise = [
    "Count",
    "Mean_Loss",
    "Median_Loss",
    "Mean_FOIF",
    "FOIF_Positive_Fraction",
    "Mean_TracIn",
    "TracIn_Positive_Fraction"
]

decile_summary_5_runs = (
    decile_results_5_runs
    .groupby(
        ["Loss_Decile", "Minority"]
    )[statistics_to_summarise]
    .agg(["mean", "std"])
    .reset_index()
)

display(decile_summary_5_runs)

Loss_Decile Minority   Count            Mean_Loss           Median_Loss  \
                           mean        std      mean       std        mean   
0            1        0  781.60  33.125519  0.000232  0.000143    0.000230   
1            1        1   23.25  36.142081  0.000264  0.000098    0.000255   
2            2        0  786.80  13.553597  0.000758  0.000358    0.000758   
3            2        1   13.00  13.711309  0.000776  0.000334    0.000792   
4            3        0  787.40   9.710819  0.001441  0.000571    0.001436   
5            3        1   12.60   9.710819  0.001490  0.000583    0.001513   
6            4        0  783.00  11.113055  0.002337  0.000797    0.002332   
7            4        1   17.20  10.963576  0.002328  0.000821    0.002354   
8            5        0  784.20   7.823043  0.003618  0.001127    0.003572   
9            5        1   15.60   8.080842  0.003652  0.001023    0.003661   
10           6        0  774.60   7.503333  0.005589  0.001708    0.005542   
11           6        1   25.40   7.503333  0.005587  0.001678    0.005611   
12           7        0  769.00   5.244044  0.008860  0.002916    0.008707   
13           7        1   31.00   5.244044  0.008929  0.002966    0.008846   
14           8        0  733.60   3.847077  0.016217  0.006425    0.015526   
15           8        1   66.40   3.847077  0.017014  0.006681    0.016938   
16           9        0  622.20  51.163464  0.041956  0.020147    0.037223   
17           9        1  177.80  51.163464  0.049158  0.023981    0.047110   
18          10        0  377.60  27.546325  0.513768  0.127786    0.208173   
19          10        1  422.40  27.546325  0.633796  0.220758    0.311833   

             Mean_FOIF           FOIF_Positive_Fraction           Mean_TracIn  \
         std      mean       std                   mean       std        mean   
0   0.000153  0.000197  0.000104               0.989318  0.019580   -0.000080   
1   0.000088  0.000482  0.000228               1.000000  0.000000    0.006587   
2   0.000354  0.000569  0.000278               0.981376  0.039488   -0.000572   
3   0.000361  0.001046  0.000538               1.000000  0.000000    0.021128   
4   0.000576  0.001013  0.000479               0.984901  0.031634   -0.000932   
5   0.000612  0.001887  0.000880               1.000000  0.000000    0.021497   
6   0.000796  0.001528  0.000732               0.984514  0.031808   -0.001472   
7   0.000862  0.002711  0.001512               1.000000  0.000000    0.022083   
8   0.001111  0.002220  0.001114               0.986306  0.026521   -0.002085   
9   0.001034  0.003826  0.001484               1.000000  0.000000    0.022993   
10  0.001680  0.003022  0.001359               0.989205  0.022017   -0.002900   
11  0.001716  0.005164  0.001885               1.000000  0.000000    0.023016   
12  0.002796  0.004098  0.001689               0.985930  0.025066   -0.003979   
13  0.003050  0.007796  0.002598               1.000000  0.000000    0.026162   
14  0.006060  0.005615  0.001654               0.972219  0.035675   -0.005463   
15  0.006567  0.012321  0.004084               1.000000  0.000000    0.026407   
16  0.018314  0.008403  0.003102               0.951510  0.021978   -0.008357   
17  0.022280  0.024372  0.008577               0.998033  0.002695    0.028822   
18  0.111929 -0.051800  0.013248               0.491988  0.198836   -0.019950   
19  0.143674  0.017463  0.012105               0.885419  0.052804    0.030130   

             TracIn_Positive_Fraction            
         std                     mean       std  
0   0.000339                 0.604425  0.317726  
1   0.007623                 0.916667  0.166667  
2   0.000765                 0.430046  0.395864  
3   0.027556                 0.911111  0.198762  
4   0.001178                 0.371156  0.415153  
5   0.029825                 0.893333  0.238514  
6   0.001926                 0.349260  0.415790  
7   0.031375                 0.896552  0.231317  
8  

In [10]:
def mean_std_string(x):
    return f"{x.mean():.6f} ± {x.std(ddof=1):.6f}"


formatted_decile_summary = (
    decile_results_5_runs
    .groupby(
        ["Loss_Decile", "Minority"]
    )[statistics_to_summarise]
    .agg(mean_std_string)
    .reset_index()
)

display(formatted_decile_summary)

,Loss_Decile,Minority,Count,Mean_Loss,Median_Loss,Mean_FOIF,FOIF_Positive_Fraction,Mean_TracIn,TracIn_Positive_Fraction
0,1,0,781.600000 ± 33.125519,0.000232 ± 0.000143,0.000230 ± 0.000153,0.000197 ± 0.000104,0.989318 ± 0.019580,-0.000080 ± 0.000339,0.604425 ± 0.317726
1,1,1,23.250000 ± 36.142081,0.000264 ± 0.000098,0.000255 ± 0.000088,0.000482 ± 0.000228,1.000000 ± 0.000000,0.006587 ± 0.007623,0.916667 ± 0.166667
2,2,0,786.800000 ± 13.553597,0.000758 ± 0.000358,0.000758 ± 0.000354,0.000569 ± 0.000278,0.981376 ± 0.039488,-0.000572 ± 0.000765,0.430046 ± 0.395864
3,2,1,13.000000 ± 13.711309,0.000776 ± 0.000334,0.000792 ± 0.000361,0.001046 ± 0.000538,1.000000 ± 0.000000,0.021128 ± 0.027556,0.911111 ± 0.198762
4,3,0,787.400000 ± 9.710819,0.001441 ± 0.000571,0.001436 ± 0.000576,0.001013 ± 0.000479,0.984901 ± 0.031634,-0.000932 ± 0.001178,0.371156 ± 0.415153
5,3,1,12.600000 ± 9.710819,0.001490 ± 0.000583,0.001513 ± 0.000612,0.001887 ± 0.000880,1.000000 ± 0.000000,0.021497 ± 0.029825,0.893333 ± 0.238514
6,4,0,783.000000 ± 11.113055,0.002337 ± 0.000797,0.002332 ± 0.000796,0.001528 ± 0.000732,0.984514 ± 0.031808,-0.001472 ± 0.001926,0.349260 ± 0.415790
7,4,1,17.200000 ± 10.963576,0.002328 ± 0.000821,0.002354 ± 0.000862,0.002711 ± 0.001512,1.000000 ± 0.000000,0.022083 ± 0.031375,0.896552 ± 0.231317
8,5,0,784.200000 ± 7.823043,0.003618 ± 0.001127,0.003572 ± 0.001111,0.002220 ± 0.001114,0.986306 ± 0.026521,-0.002085 ± 0.002881,0.316100 ± 0.412868
9,5,1,15.600000 ± 8.080842,0.003652 ± 0.001023,0.003661 ± 0.001034,0.003826 ± 0.001484,1.000000 ± 0.000000,0.022993 ± 0.030347,0.861538 ± 0.309609


In [11]:
# density_df["Loss_Decile"] = pd.qcut(
#     density_df["Training_Loss"],
#     q=10,
#     labels=False,
#     duplicates="drop"
# ) + 1

In [12]:
# decile_label_summary = (
#     density_df
#     .groupby(["Loss_Decile", "Minority"])
#     .agg(
#         Count=("Train_ID", "size"),
#         Mean_Loss=("Training_Loss", "mean"),

#         Mean_FOIF=("FOIF_Score", "mean"),
#         FOIF_Positive_Fraction=(
#             "FOIF_Score",
#             lambda x: (x > 0).mean()
#         ),

#         Mean_TracIn=("TracIn_Score", "mean"),
#         TracIn_Positive_Fraction=(
#             "TracIn_Score",
#             lambda x: (x > 0).mean()
#         )
#     )
#     .reset_index()
# )

# print(decile_label_summary)